Cell 1: Imports & Load Data

In [1]:
import pandas as pd
import numpy as np
import time
from rank_bm25 import BM25Okapi

# Load clean books dataset
df = pd.read_csv("../data/books_cleaned.csv")
print(f"Loaded {len(df)} cleaned books for BM25 Keyword Indexing!")

Loaded 50000 cleaned books for BM25 Keyword Indexing!


Cell 2: Tokenization & Indexing (How BM25 processes text)

In [2]:
print("⏳ Tokenizing text into words for BM25 Keyword Search...")
start_time = time.time()

# Lowercase and split text into individual word tokens
corpus_tokens = [
    str(text).lower().split() for text in df['combined_text']
]

# Initialize BM25 algorithm
bm25 = BM25Okapi(corpus_tokens)

elapsed = time.time() - start_time
print(f"✅ BM25 Index successfully built for {len(df)} books in {elapsed:.2f} seconds!")
print(f"📝 Sample Tokenized Book #1:\n{corpus_tokens[0][:15]}...")


⏳ Tokenizing text into words for BM25 Keyword Search...
✅ BM25 Index successfully built for 50000 books in 0.60 seconds!
📝 Sample Tokenized Book #1:
['title:', 'the', 'hunger', 'games', '(the', 'hunger', 'games,', '#1)', '|', 'author:', 'suzanne', 'collins', '|', 'published:', '2008']...


Cell 3: Interactive BM25 Keyword Search Function

In [3]:
def search_bm25_interactive(query_text, top_k=5):
    start_time = time.time()
    
    # Tokenize user query
    tokenized_query = str(query_text).lower().split()
    
    # Calculate BM25 scores for all books
    doc_scores = bm25.get_scores(tokenized_query)
    
    # Get top_k indices
    top_indices = doc_scores.argsort()[::-1][:top_k]
    
    elapsed_ms = (time.time() - start_time) * 1000
    
    # Build results DataFrame
    results = []
    for idx in top_indices:
        score = doc_scores[idx]
        if score > 0:
            row = df.iloc[idx].to_dict()
            row['bm25_score'] = round(float(score), 4)
            results.append(row)
            
    res_df = pd.DataFrame(results)
    print(f"⚡ BM25 Keyword Search for '{query_text}' returned {len(res_df)} matches in {elapsed_ms:.2f} ms")
    
    if len(res_df) > 0:
        return res_df[['title', 'authors', 'genres', 'average_rating', 'bm25_score']]
    else:
        return "No exact keyword matches found."

# Test exact author search:
search_bm25_interactive("Suzanne Collins Hunger Games", top_k=5)


⚡ BM25 Keyword Search for 'Suzanne Collins Hunger Games' returned 5 matches in 51.43 ms


,title,authors,genres,average_rating,bm25_score
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,General Fiction,4.34,40.2328
1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,General Fiction,4.34,40.2328
2,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,General Fiction,4.34,40.2328
3,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,General Fiction,4.34,40.2328
4,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,General Fiction,4.34,40.2328


Cell 4: Side-by-Side Comparison: Vector Search vs BM25 Search

In [4]:
# Let's compare how BM25 handles exact names vs how Vector Search handles themes
print("🔍 TEST 1: Exact Author Match ('J.R.R. Tolkien')")
display(search_bm25_interactive("J.R.R. Tolkien Hobbit Lord Rings", top_k=3))

print("\n🔍 TEST 2: General Concept Query ('artificial intelligence robot future')")
display(search_bm25_interactive("artificial intelligence robot future", top_k=3))


🔍 TEST 1: Exact Author Match ('J.R.R. Tolkien')
⚡ BM25 Keyword Search for 'J.R.R. Tolkien Hobbit Lord Rings' returned 3 matches in 98.79 ms


,title,authors,genres,average_rating,bm25_score
0,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,J.R.R. Tolkien,Fantasy & Magic,4.59,47.7545
1,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,J.R.R. Tolkien,Fantasy & Magic,4.59,47.7545
2,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,J.R.R. Tolkien,Fantasy & Magic,4.59,47.7545



🔍 TEST 2: General Concept Query ('artificial intelligence robot future')
⚡ BM25 Keyword Search for 'artificial intelligence robot future' returned 3 matches in 48.29 ms


,title,authors,genres,average_rating,bm25_score
0,"I, Robot (Robot #0.1)",Isaac Asimov,Sci-Fi & Dystopian,4.17,12.3772
1,"I, Robot (Robot #0.1)",Isaac Asimov,Sci-Fi & Dystopian,4.17,12.3772
2,"I, Robot (Robot #0.1)",Isaac Asimov,Sci-Fi & Dystopian,4.17,12.3772
